In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from catboost import Pool, CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import SelectKBest, VarianceThreshold

try:
    import pyarrow
    print("PyArrow is installed. Version:", pyarrow.__version__)
except ModuleNotFoundError:
    print("PyArrow is NOT found in this environment. You installed it in a different one.")

PyArrow is installed. Version: 23.0.1


In [2]:
train_main = pd.read_parquet('../data/train_main_features.parquet')
test_main = pd.read_parquet('../data/test_main_features.parquet')
target = pd.read_parquet('../data/train_target.parquet')
train_extra = pd.read_parquet('../data/train_extra_features.parquet')
test_extra = pd.read_parquet('../data/test_extra_features.parquet')


print('Тренировочные данные:', train_main.shape)
print('Тестовые данные:', test_main.shape)
print('Тренировочные данные:', train_extra.shape)
print('Тестовые данные:', test_extra.shape)

Тренировочные данные: (750000, 200)
Тестовые данные: (250000, 200)
Тренировочные данные: (750000, 2242)
Тестовые данные: (250000, 2242)


In [3]:
cat_feature_names = [
    col_name for col_name in train_main.columns 
    if col_name.startswith("cat_feature")
]

In [4]:
train = pd.merge(train_main, train_extra, on="customer_id", how="left")
test  = pd.merge(test_main,  test_extra,  on="customer_id", how="left")

In [5]:
train[cat_feature_names] = train[cat_feature_names].astype(str)
test[cat_feature_names] = test[cat_feature_names].astype(str)

In [6]:
train_pool = Pool(data = train.drop("customer_id", axis=1), 
                  label = target.drop("customer_id", axis=1), 
                  cat_features = cat_feature_names)

In [7]:
train_cat = train[cat_feature_names]

убираем признаки с низкой дисперсией (константные)

In [8]:
selector = VarianceThreshold(threshold=1e-5)
train_reduced = selector.fit_transform(train)

In [9]:
selected_columns = selector.get_feature_names_out(train.columns)


убрали почти 200 мусорных признаков

In [10]:
print(train_reduced.shape[1], train.shape[1])

2271 2441


In [11]:
train_reduced = pd.DataFrame(train_reduced, columns = selected_columns)

преобразуем массив в дф

In [12]:
train_reduced.head()

,customer_id,cat_feature_1,cat_feature_2,cat_feature_3,cat_feature_4,cat_feature_5,cat_feature_6,cat_feature_7,cat_feature_8,cat_feature_9,...,num_feature_2359,num_feature_2362,num_feature_2363,num_feature_2364,num_feature_2365,num_feature_2368,num_feature_2369,num_feature_2370,num_feature_2371,num_feature_2372
0,1000001,1.0,0.0,2.0,1.0,2.0,3.0,2.0,2.0,4.0,...,-0.023246,-0.020769,-0.14374,NaN,-0.00276,-0.030607,-0.015407,-0.404816,-0.395589,NaN
1,1000002,1.0,0.0,0.0,1.0,0.0,3.0,0.0,0.0,0.0,...,-0.023246,-0.020769,-0.14374,NaN,-0.001345,-0.030607,-0.047964,1.630249,-0.395589,-0.201813
2,1000003,1.0,0.0,0.0,1.0,0.0,3.0,0.0,0.0,4.0,...,-0.023246,-0.020769,-0.14374,NaN,-0.00276,-0.030607,-0.045637,-0.404816,-0.395589,-0.271623
3,1000004,1.0,0.0,2.0,1.0,2.0,3.0,2.0,2.0,3.0,...,-0.023246,-0.020769,-0.14374,NaN,-0.00276,-0.030607,-0.058314,-0.404816,NaN,NaN
4,1000005,1.0,2.0,0.0,1.0,0.0,3.0,0.0,0.0,2.0,...,NaN,NaN,NaN,NaN,-0.00276,-0.030607,-0.043266,-0.404816,0.048602,0.206248


уберем столбцы, где 80% пропущенные значения

In [13]:
# threshold = 0.93
# train_less_none = train_reduced.dropna(thresh=int((1 - threshold) * len(train_reduced)), axis = 1)

In [14]:
# train_less_none.shape[1]

In [16]:
train_reduced.to_parquet("../modified_data/train_no_nan_min_var.parquet")

2267 2271 2441
